# openCypher cheat sheet

This notebook is a reference for the openCypher used in the workshop, and every example runs against
the workshop's supplier graph in Amazon Neptune. It's worth keeping open in a second tab, and any
cell can be run to see what it returns.

**How the cells work.** Each example is a `%%oc` cell, which runs the query in Neptune and shows the
result. Choose the **Table** tab for rows, or the **Graph** tab when the query returns a path
(`RETURN p`). `-d name` labels each vertex with its `name` property in the graph view, and
`-de share_pct` puts each ownership stake on its edge. There's no Connect cell, because the notebook
instance is already connected to the workshop's Neptune cluster.

**Values come in as parameters.** The examples never paste a name into the query. They write
`$supplier_name` and the like, and `-qp CHEAT_PARAMS` passes the values from the dict in the first
cell. That's the practice to follow in your own code, because Neptune caches one plan for a
parameterised query and reuses it for any value, and no value can change the query's meaning. Change
a value in the first cell and run an example again to ask about another supplier.

**Reading is safe, and writing is fenced off.** The read examples change nothing. The write examples
only touch vertices labelled `CheatSheet`, and the last cell deletes them. Don't point a write at the
workshop's own labels (`Supplier`, `Company` and so on), or the session's results will change.

| Section | What's in it |
|---|---|
| 1 · Match a pattern | vertices by label and property, edges in both directions, paths, optional matches, existence checks |
| 2 · Walk any depth | fixed and variable-length hops, the pattern behind every graph rule in the workshop |
| 3 · Return values | properties, aliases, distinct values, ordering, counting, paging, unions |
| 4 · Aggregate, and work with lists | `WITH`, `sum()`, `collect()`, list comprehensions, "all elements match", set difference |
| 5 · Subqueries and the nearest path | `CALL { }` once per row, and the nearest match without `shortestPath()` |
| 6 · Parameters and query plans | `$name` parameters, and how Neptune ran a query |
| 7 · Write, safely | `CREATE`, `MERGE`, `SET`, `REMOVE`, `DELETE` on a scratch label |
| 8 · Operators and functions | the ones that run on this cluster |

The data model, for reference: `(Supplier)-[:OWNED_BY {share_pct}]->(Company)-[:OWNED_BY]->(Company)`
up to `(:Company)-[:SANCTIONED_BY]->(:SanctionsList)`; `(Supplier)-[:SOURCES_FROM]->(Company)` down to
`(:Company)-[:ON_LIST]->(:RestrictedList)`; and `APPROVED_FOR` (to `Product`), `OPERATES_IN` (to
`Region`), `HOLDS_CERT` (to `Certification`), `HAS_WAREHOUSE` (to `Warehouse`).

In [ ]:
# The parameters every example below uses, passed to each %%oc cell with -qp CHEAT_PARAMS
CHEAT_PARAMS = {
    "supplier_name": "AnyCompany Components",          # the supplier on the order
    "other_supplier": "AnyCompany Electronics",       # the one with a 20% sanctioned owner
    "company_name": "AnyCompany Components Holdings BV",
    "region_name": "US-East",
    "suppliers": ["AnyCompany Components", "AnyCompany Supply Co", "AnyCompany Devices"],
    "ofac_threshold": 50,                               # as in queries/rule_settings.json
}

## 1 · Match a pattern

`MATCH` describes a shape, with vertices in parentheses and edges in square brackets, and an arrow
for the direction. Neptune returns every place in the graph where the shape fits.

#### Vertices with a label

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (r:Region) RETURN r.name, r.country ORDER BY r.name

#### A vertex by label and property

The property map in the pattern is shorthand for a `WHERE` clause; the next cell is the same query.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name}) RETURN s

#### The same, with WHERE

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier) WHERE s.name = $supplier_name RETURN s.name, s.status

#### Follow an edge out

`-[o:OWNED_BY]->` names the edge `o`, so you can read its properties.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name})-[o:OWNED_BY]->(owner)
RETURN owner.name, o.share_pct

#### Follow an edge in

The arrow points the other way, so this finds who is owned *by* Holdings BV.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (c:Company {name: $company_name})<-[o:OWNED_BY]-(owned)
RETURN owned.name, o.share_pct

#### Return a path, and draw it

Choose the **Graph** tab.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:HAS_WAREHOUSE]->(:Warehouse)-[:LOCATED_IN]->(:Region)
RETURN p

#### An optional match

`OPTIONAL MATCH` keeps the row when the pattern is missing, and the missing pieces come back as `null`, the way a SQL `LEFT JOIN` does.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier) WHERE s.name IN $suppliers
OPTIONAL MATCH (s)-[:HOLDS_CERT {cert_status: 'active'}]->(c:Certification)
RETURN s.name, collect(c.code) AS active_certs

#### Whether a pattern exists

A pattern in `WHERE` tests for existence without returning it. `WHERE NOT (s)-[...]->(...)` tests for absence.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier)
WHERE (s)-[:OWNED_BY]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
RETURN count(s) AS directly_owned_by_a_listed_company

## 2 · Walk any depth

A `*` on an edge repeats it. This is what makes a graph the right tool for ownership and supply
chains, because the query says what to follow rather than how deep the chain is. openCypher never
follows the same edge twice in one path, so an unbounded walk always ends. The workshop's walks are
unbounded because we know the dataset, whose deepest chain is eight ownership hops. In production,
an unbounded walk's cost depends on the data, so consider an upper bound such as `*1..12`, chosen
from the depth of your own data.

#### Exactly two hops

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name})-[:OWNED_BY*2]->(grandparent)
RETURN DISTINCT grandparent.name

#### Between one and three hops

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:OWNED_BY*1..3]->(owner)
RETURN owner.name, length(p) AS hops ORDER BY hops

#### One hop or more, as far as the chain goes

The walk from Cell 2.1, up to the OFAC list. Choose the **Graph** tab.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
RETURN p

#### At most two hops

`*..2` means one or two hops, which reaches the supplier's makers and their makers.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name})-[:SOURCES_FROM*..2]->(maker)
RETURN DISTINCT maker.name

## 3 · Return values

`RETURN` says what comes back. Everything after it works the way it does in SQL.

#### A property, with an alias

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (r:Region) RETURN r.name AS region, r.country AS country

#### Distinct values

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (r:Region) RETURN DISTINCT r.country

#### Ordered, ascending and descending

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (:Region {name: $region_name})-[t:SHIPS_TO]->(r:Region)
RETURN r.name, t.transit_days ORDER BY t.transit_days ASC, r.name DESC

#### Count

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier) RETURN count(s) AS suppliers

#### Limit, and page

`SKIP 10 LIMIT 5` is the third page of five.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier) RETURN s.name ORDER BY s.name SKIP 10 LIMIT 5

#### Combine two queries: UNION and UNION ALL

`UNION` drops duplicate rows; `UNION ALL` keeps them. Both queries must return the same columns.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (:Supplier {name: $supplier_name})-[:OWNED_BY]->(o) RETURN o.name AS owner
UNION
MATCH (:Supplier {name: $other_supplier})-[:OWNED_BY]->(o) RETURN o.name AS owner

## 4 · Aggregate, and work with lists

`WITH` passes rows from one part of a query to the next, and is where you aggregate, filter the
aggregate, or sort before going on.

#### Sum per vertex, then filter the sum

This is the shape of the 50% rule (Cell 2.3), in which only stakes held by blocked owners are summed.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (e)-[o:OWNED_BY]->(owner:Company) WHERE owner.blocked = true
WITH e, sum(o.share_pct) AS blocked_share
WHERE blocked_share >= $ofac_threshold
RETURN e.name, blocked_share ORDER BY blocked_share DESC LIMIT 10

#### Collect rows into a list

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name})-[:APPROVED_FOR]->(p:Product)
RETURN collect(p.name) AS approved_for, count(p) AS n

#### A path as a list of names

A list comprehension, `[x IN list | expression]`, is how the report builds its evidence.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:SOURCES_FROM*1..]->(:Company)-[:ON_LIST]->(:RestrictedList)
RETURN [n IN nodes(p) | n.name] AS evidence, length(p) AS hops

#### "Every element matches", without all()

This cluster's engine (1.4.8.0) has no `all()`, so count the elements that match and compare with the length. `tail()` drops the first element, the supplier.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
RETURN [n IN nodes(p) | n.name] AS chain,
       size([n IN tail(nodes(p)) WHERE n.blocked = true]) = length(p) - 1 AS every_owner_blocked

#### Set difference of two lists

`collSubtract()` is a Neptune function that returns what's in the first list and not in the second.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier {name: $supplier_name})
OPTIONAL MATCH (:Region {name: $region_name})-[:REQUIRES_CERT]->(rc:Certification)
WITH s, collect(rc.code) AS required
OPTIONAL MATCH (s)-[:HOLDS_CERT {cert_status: 'active'}]->(hc:Certification)
WITH required, collect(hc.code) AS held
RETURN required, held, collSubtract(required, held) AS missing

## 5 · Subqueries and the nearest path

`CALL { WITH x ... RETURN y }` runs a subquery once for each incoming row. `ORDER BY ... LIMIT` inside
it applies per row, which is how the report picks one piece of evidence per rule.

#### A subquery once per row

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (s:Supplier) WHERE s.name IN $suppliers
CALL {
  WITH s
  OPTIONAL MATCH tiers = (s)-[:SOURCES_FROM*1..]->(:Company)-[:ON_LIST]->(:RestrictedList)
  RETURN tiers ORDER BY length(tiers) LIMIT 1
}
RETURN s.name, length(tiers) - 1 AS uflpa_tier

#### The nearest match, without shortestPath()

Neptune's openCypher has no `shortestPath()`; order the paths by length and keep the first.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH p = (s:Supplier {name: $supplier_name})-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
RETURN [n IN nodes(p) | n.name] AS chain ORDER BY length(p) LIMIT 1

## 6 · Parameters and query plans

Every example in this notebook already uses parameters. This section shows the two things they buy.
The same query text runs for other values, and `%%oc explain` shows how Neptune planned it.

In [ ]:
# A Python cell: the same parameters, for another supplier
DEVICES = dict(CHEAT_PARAMS, supplier_name="AnyCompany Devices")

#### The same query, other values

The query text is unchanged, so Neptune reuses its cached plan.

In [ ]:
%%oc -qp DEVICES
MATCH (s:Supplier {name: $supplier_name})-[:OPERATES_IN]->(r:Region)
RETURN s.name, collect(r.name) AS regions

#### How Neptune ran it

`%%oc explain` shows the plan instead of the rows, including which step ran first and how many rows each step produced.

In [ ]:
%%oc explain -qp CHEAT_PARAMS
MATCH (s:Supplier {name: $supplier_name})-[:OWNED_BY*1..]->(c:Company)
RETURN count(c)

## 7 · Write, safely

These cells only touch vertices labelled `CheatSheet`, and the last one deletes them all. Run them in
order.

#### Create vertices, and an edge between them

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
CREATE (a:CheatSheet {name: 'demo-a'})-[:LINKS_TO {weight: 1}]->(b:CheatSheet {name: 'demo-b'})
RETURN a.name, b.name

#### Create several from a list

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
UNWIND [{name: 'demo-c'}, {name: 'demo-d'}] AS props
CREATE (n:CheatSheet) SET n = props
RETURN n.name

#### Merge: match it, or create it

`demo-a` already exists, so this matches it and increments `seen`. Run it again to watch the count go up.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MERGE (n:CheatSheet {name: 'demo-a'})
  ON CREATE SET n.created = true
  ON MATCH SET n.seen = coalesce(n.seen, 0) + 1
RETURN n.name, n.seen, n.created

#### Set, and remove, a property

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (n:CheatSheet {name: 'demo-b'})
SET n.colour = 'blue'
REMOVE n.colour
RETURN n

#### Delete an edge

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (:CheatSheet)-[r:LINKS_TO]->(:CheatSheet)
DELETE r

#### Delete the scratch vertices, and any edges they still have

`DETACH DELETE` removes a vertex with its edges; plain `DELETE` fails on a vertex that still has edges.

In [ ]:
%%oc -qp CHEAT_PARAMS -d name -de share_pct -l 40
MATCH (n:CheatSheet)
DETACH DELETE n

## 8 · Operators and functions

| Type | Operators |
|---|---|
| General | `DISTINCT`, `x.y` (property access) |
| Math | `+`, `-`, `*`, `/`, `%` (no `^` on this cluster: use `exp()` and `log()`) |
| Comparison | `=`, `<>`, `<`, `>`, `<=`, `>=`, `IS NULL`, `IS NOT NULL` |
| Boolean | `AND`, `OR`, `NOT`, `XOR` |
| String | `STARTS WITH`, `ENDS WITH`, `CONTAINS`, `+` (concatenate) |
| List | `+` (concatenate), `IN`, `[i]` (index), `[i..j]` (slice) |

| Type | Functions |
|---|---|
| Scalar | `coalesce()`, `head()`, `last()`, `length()`, `size()`, `type()`, `id()`, `properties()`, `startNode()`, `endNode()`, `toString()`, `toInteger()`, `toFloat()`, `toBoolean()` |
| Aggregating | `count()`, `sum()`, `avg()`, `min()`, `max()`, `collect()` |
| List | `nodes()`, `relationships()`, `labels()`, `keys()`, `range()`, `reverse()`, `tail()`, `collSubtract()` (Neptune) |
| Math | `abs()`, `ceil()`, `floor()`, `round()`, `sign()`, `rand()`, `sqrt()`, `exp()`, `log()`, `log10()` |
| String | `toUpper()`, `toLower()`, `trim()`, `lTrim()`, `rTrim()`, `replace()`, `split()`, `substring()`, `left()`, `right()`, `reverse()` |

The next cell calls one function from each row, so you can see they run on this cluster. For the full
list and the differences from other openCypher engines, see
[openCypher in Neptune](https://docs.aws.amazon.com/neptune/latest/userguide/access-graph-opencypher.html).

In [ ]:
%%oc -qp CHEAT_PARAMS
MATCH p = (s:Supplier {name: $supplier_name})-[o:OWNED_BY]->(c:Company)
RETURN coalesce(s.missing, 'default') AS coalesce_, size(c.name) AS size_, type(o) AS type_,
       toString(o.share_pct) + '%' AS toString_, toInteger('42') AS toInteger_,
       count(c) AS count_, nodes(p)[0].name AS nodes_, labels(c) AS labels_,
       range(1, 3) AS range_, tail([1, 2, 3]) AS tail_, collSubtract([1, 2, 3], [2]) AS collSubtract_,
       round(2.567) AS round_, sqrt(16) AS sqrt_, abs(-3) AS abs_,
       toUpper('neptune') AS toUpper_, split('a,b', ',') AS split_, substring('neptune', 0, 3) AS substring_,
       7 % 3 AS modulo_, 'nep' + 'tune' AS concat_, [1, 2, 3][1..] AS slice_